# Telecom Egypt Intelligent Assistant (Self-Contained Groq Version)

This notebook contains the complete source code for the Telecom Egypt RAG pipeline, making it fully self-contained. You can run this directly on **Google Colab** or **Kaggle** without needing to upload the rest of the python scripts from the repository.

It uses the **Groq API** for fast, high-quality generation.

## 1. Install Dependencies

In [15]:
!pip install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers transformers edge-tts pypdf python-docx docx2txt pillow beautifulsoup4 requests langchain-groq groq

In [16]:
!pip install docx2txt

## 2. Setup API Key and Logging
Get your API key from the [Groq Console](https://console.groq.com/).

In [17]:
  import os
import requests
from bs4 import BeautifulSoup
from typing import List
import logging
from PIL import Image
from getpass import getpass

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Please enter your Groq API Key:")
os.environ["GROQ_API_KEY"] = getpass()

Please enter your Groq API Key:
··········


## 3. Define Prompts

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

RAG_SYSTEM_PROMPT = """You are a helpful and intelligent assistant for Telecom Egypt (WE).
Your primary task is to answer the user's question based on the provided context and the conversation history.

Follow these STRICT rules:
1. Answer the question using ONLY the provided context and the information from the conversation history (including uploaded documents). If the answer cannot be found in these sources, say "I do not have enough information to answer that based on the provided context." Do not use your own external knowledge.
2. CRITICAL LANGUAGE RULE: YOU MUST DETECT THE USER'S LANGUAGE AND REPLY IN THE EXACT SAME LANGUAGE. If the user's question is in English, your ENTIRE final response MUST be in English (even if the retrieved context is in Arabic). If the user asks in Arabic, you MUST reply in Arabic. NO EXCEPTIONS.
3. DO NOT include source citations inside your response. Provide a natural and continuous answer.
4. REASONING: Before providing your final answer, deeply think step-by-step inside <think>...</think> tags. Break down indirect, vague, or complex questions to understand the user's true intent, and carefully map it to the provided context before answering.
5. DOCUMENT LANGUAGE: If the user uploads a document without typing a specific question, you MUST detect the language of the text inside the document. If the document is in Arabic, you MUST reply in Arabic, summarizing or answering based on the document.

Context:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", RAG_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])


## 4. Document Scrapers and Loaders

In [19]:
import base64
from groq import Groq
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader

def process_image_ocr(file_path: str) -> List[Document]:
    try:
        with open(file_path, "rb") as image_file:
            base64_image = base64.b64encode(image_file.read()).decode('utf-8')

        client = Groq()
        logger.info(f"Running high-quality OCR via Groq Vision for {file_path}...")
        completion = client.chat.completions.create(
            model="llama-3.2-90b-vision-instruct",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Extract all the text from this image exactly as written. Ensure Arabic and English text is captured perfectly. Output ONLY the extracted text, with absolutely no conversational filler or commentary."
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0,
            max_completion_tokens=2048,
        )
        text = completion.choices[0].message.content
        return [Document(page_content=text, metadata={"source": file_path, "type": "image_ocr"})]
    except Exception as e:
        logger.error(f"Error processing image {file_path}: {e}")
        return []

def load_document(file_path: str) -> List[Document]:
    if not os.path.exists(file_path):
        logger.error(f"File not found: {file_path}")
        return []

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == '.pdf': return PyPDFLoader(file_path).load()
        elif ext == '.docx': return Docx2txtLoader(file_path).load()
        elif ext == '.txt': return TextLoader(file_path, encoding='utf-8').load()
        elif ext in ['.png', '.jpg', '.jpeg']: return process_image_ocr(file_path)
        else:
            logger.warning(f"Unsupported file extension: {ext}")
            return []
    except Exception as e:
        logger.error(f"Error loading {file_path}: {e}")
        return []

def scrape_te_page(url: str) -> List[Document]:
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        response.encoding = 'utf-8'

        soup = BeautifulSoup(response.text, 'html.parser')
        for script in soup(["script", "style", "header", "footer", "nav"]):
            script.decompose()

        text = soup.get_text(separator=' ', strip=True)
        logger.info(f"Successfully scraped {len(text)} characters from {url}")
        return [Document(page_content=text, metadata={"source": url, "type": "web_page"})]
    except Exception as e:
        logger.error(f"Error scraping {url}: {e}")
        return []


## 5. Vector Store Configuration

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def get_embeddings_model() -> HuggingFaceEmbeddings:
    return HuggingFaceEmbeddings(
        model_name=EMBEDDING_MODEL_NAME,
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )

def setup_vector_store(documents: List[Document], persist_directory: str = "./data/chroma_db") -> Chroma:
    if not documents:
        return Chroma(collection_name="te_knowledge_base", embedding_function=get_embeddings_model(), persist_directory=persist_directory)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)
    chunks = text_splitter.split_documents(documents)
    logger.info(f"Split documents into {len(chunks)} chunks.")

    os.makedirs(persist_directory, exist_ok=True)
    return Chroma.from_documents(
        documents=chunks,
        embedding=get_embeddings_model(),
        persist_directory=persist_directory,
        collection_name="te_knowledge_base"
    )

## 6. Run Ingestion (Build Knowledge Base)

In [21]:
# List of high-value TE knowledge targets
faq_urls = [
    "https://www.te.eg/about-te/faq",                  # Main Mobile & USSD FAQs
    "https://te.eg/en/about-te/faq/fixed-broadband",   # Home Internet (WE Space, Routers, Quotas)
    "https://te.eg/en/about-te/faq/fixed-voice"        # Landline (Billing, Installments, Tariffs)
]

all_docs = []
for url in faq_urls:
    logger.info(f"Processing knowledge target: {url}")
    # Using our updated Jina AI scraper from Section 4
    docs = scrape_te_page(url)
    all_docs.extend(docs)

# Set up the Chroma DB and generate embeddings from all scraped FAQ pages
vector_db = setup_vector_store(all_docs, persist_directory="./data/chroma_db")
logger.info(f"Knowledge base successfully populated with {len(all_docs)} source pages!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 7. RAG Pipeline Implementation (Groq)

In [22]:
# --- 4. RAG Pipeline (UPGRADED WITH QUERY REWRITER) ---

from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
import re
def format_docs(docs):
    return "\n\n".join(f"[Source: {d.metadata.get('source', 'Unknown')}]\nContent: {d.page_content}" for d in docs)

class GroqRAGPipeline:
    def __init__(self, model_name: str = "llama-3.3-70b-versatile"):
        self.vector_store = Chroma(
            collection_name="te_knowledge_base",
            embedding_function=get_embeddings_model(),
            persist_directory="./data/chroma_db"
        )
        self.retriever = self.vector_store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 6, "fetch_k": 20}
        )
        self.llm = ChatGroq(model_name=model_name, temperature=0.1)

        # Notice we now pass 'formatted_context' directly from our smart retriever
        self.rag_chain = (
            {
                "context": lambda x: x["formatted_context"],
                "uploaded_context": lambda x: x.get("uploaded_context", "No document currently uploaded."),
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"]
            }
            | qa_prompt
            | self.llm
            | StrOutputParser()
        )

    def _rewrite_query(self, user_input: str, chat_history: list) -> str:
        """Translates conversational follow-ups into standalone search queries."""
        if not chat_history:
            return user_input

        rewrite_messages = [
            SystemMessage(content="You are a search query optimizer. Look at the chat history and the latest user question. If the latest question relies on context or pronouns from the history (e.g., 'how much is it?', 'what are the speeds?'), rewrite it into a self-contained, standalone search query that includes the actual product names or subjects mentioned earlier. If it is already standalone, return it unchanged. Output ONLY the final search query without explanations, tags, or quotation marks.")
        ]
        # Pass the last 4 messages (2 turns) of conversation history for context
        rewrite_messages.extend(chat_history[-4:])
        rewrite_messages.append(HumanMessage(content=f"Latest Question: {user_input}"))

        try:
            rewritten = self.llm.invoke(rewrite_messages).content.strip()
            # Clean up any accidental think tags generated by reasoning models
            rewritten = re.sub(r"<think.*?>.*?</think>", "", rewritten, flags=re.DOTALL | re.IGNORECASE).strip()
            logger.info(f"Original query: '{user_input}' -> Rewritten for ChromaDB: '{rewritten}'")
            return rewritten
        except Exception as e:
            logger.error(f"Query rewrite error: {e}")
            return user_input

    def stream_query(self, user_input: str, chat_history: list = None, uploaded_context: str = ""):
        chat_history = chat_history or []

        # 1. REWRITE THE QUERY: Turns "How much is it?" into "How much is WE Space?"
        search_query = self._rewrite_query(user_input, chat_history)

        # 2. RETRIEVE DOCS: Search ChromaDB using the smart, standalone query
        docs = self.retriever.invoke(search_query)
        formatted_docs = format_docs(docs)

        # 3. STREAM ANSWER: Feed the retrieved docs and original chat flow to the LLM
        stream = self.rag_chain.stream({
            "input": user_input, # Keep original input so conversation sounds natural
            "chat_history": chat_history,
            "uploaded_context": uploaded_context or "No document currently uploaded.",
            "formatted_context": formatted_docs
        })
        for chunk in stream:
            yield chunk, docs

pipeline = GroqRAGPipeline(model_name="llama-3.3-70b-versatile")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 8. Query the Pipeline

In [23]:
query = "What services does Telecom Egypt offer for personal use?"

print("\nTE Assistant:")
print("-" * 60)

full_answer = ""
retrieved_docs = []

# Iterate through the generator
for chunk, docs in pipeline.stream_query(query):
    print(chunk, end="", flush=True)
    full_answer += chunk
    retrieved_docs = docs

print("\n" + "-" * 60)

if retrieved_docs:
    print("\nSources:")
    sources = set(doc.metadata.get('source', 'Unknown') for doc in retrieved_docs)
    for source in sources:
        print(f"  • {source}")

# Save as a dictionary so Section 10 can use it later
result = {"answer": full_answer, "context": retrieved_docs}


TE Assistant:
------------------------------------------------------------
<think> To answer this question, we need to consider the context provided and identify the services that Telecom Egypt (WE) offers for personal use. The context includes various FAQs about WE's services, including mobile, internet, and fixed line services. We should look for services that are relevant to personal use, such as mobile plans, internet packages, and other related services. </think>

Telecom Egypt (WE) offers various services for personal use, including mobile services, internet services, and fixed line services. The mobile services include different plans such as WE Gold 260, WE Gold 525, WE Gold 775, WE Gold 1050, WE Gold 1,300, and WE Gold 2,000. The internet services include fixed broadband plans with different speeds and quotas. Additionally, WE offers other services such as "Salfny" which allows users to borrow credit when their balance is low, and "Kalemni" which allows users to send a messag

## 9. Speech Integration (ASR & TTS)

In [24]:
import subprocess
import logging
import os
from groq import Groq

logger = logging.getLogger(__name__)

def get_clean_path(fp):
    """Safely extracts the file path from Gradio file objects, audio dicts, or strings."""
    if not fp:
        return ""
    if isinstance(fp, dict):
        return fp.get("path") or fp.get("name") or str(fp)
    elif hasattr(fp, "path") and getattr(fp, "path"):
        return getattr(fp, "path")
    elif hasattr(fp, "name") and getattr(fp, "name"):
        return getattr(fp, "name")
    return str(fp)

class EgyptianASR:
    def __init__(self):
        self.client = Groq()

    def transcribe(self, audio_path) -> str:
        clean_path = get_clean_path(audio_path)
        if not clean_path or not os.path.exists(clean_path):
            logger.error(f"ASR Error: File not found or invalid path -> '{clean_path}'")
            return ""

        try:
            logger.info(f"Transcribing audio via Groq Whisper: {clean_path}")
            with open(clean_path, "rb") as file:
                transcription = self.client.audio.transcriptions.create(
                    file=(os.path.basename(clean_path), file.read()),
                    model="whisper-large-v3-turbo",
                    prompt="This is a customer service conversation. هذه محادثة لخدمة العملاء باللهجة المصرية.",
                    temperature=0.0,
                    response_format="json"
                )
            return getattr(transcription, "text", str(transcription))
        except Exception as e:
            logger.error(f"ASR Processing Error: {e}")
            return ""

class HighQualityTTS:
    def __init__(self, voice="ar-EG-SalmaNeural"):
        self.voice = voice

    def synthesize(self, text: str, output_path: str):
        logger.info(f"Synthesizing text using Edge TTS ({self.voice})")
        subprocess.run(
            ["edge-tts", "--voice", self.voice, "--text", text, "--write-media", output_path],
            check=True
        )

## 10. Test Audio Generation

In [25]:
from IPython.display import Audio

try:
    # Optional: Test ASR (Requires an uploaded audio file like "test_audio.wav")
    # asr = EgyptianASR()
    # transcript = asr.transcribe("test_audio.wav")
    # print(transcript)

    # Initialize TTS and synthesize the result
    tts = HighQualityTTS(voice="ar-EG-SalmaNeural")
    audio_file = "response_output.mp3"

    # Generate audio (using Colab's existing event loop)
    tts.synthesize(result["answer"], audio_file)

    logger.info("Generated audio successfully.")
except Exception as e:
    logger.error(f"Audio processing failed: {e}")

# Audio(audio_file) # Uncomment to play in notebook


## 11. Install Gradio

In [26]:
!pip install gradio


In [27]:
!pip install --upgrade gradio

## 12. Run the Interactive Frontend in the Notebook

In [14]:
import gradio as gr
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import re
import uuid
import os
import base64

def wrap_rtl(text):
    if re.search("[؀-ۿ]", text):
        return f"<div dir='rtl' style='text-align: right;'>\n\n{text}\n\n</div>"
    return text

def process_interaction(audio_filepath, file_paths, text_input, history):
    uploaded_context = ""
    if file_paths:
        from langchain_text_splitters import RecursiveCharacterTextSplitter
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200, length_function=len)

        if isinstance(file_paths, (dict, str)):
            file_paths = [file_paths]
        elif not isinstance(file_paths, (list, tuple)):
            file_paths = [file_paths]

        for fp in file_paths:
            actual_path = get_clean_path(fp)
            docs = load_document(actual_path)
            if docs:
                for d in docs:
                    text_content = d.page_content.strip() if d.page_content else ""
                    if text_content: uploaded_context += f"\n[Uploaded Document]: {text_content}\n"
                chunks = text_splitter.split_documents(docs)
                if chunks: pipeline.vector_store.add_documents(chunks)

    # Sanitize and transcribe voice recording safely
    if audio_filepath:
        asr = EgyptianASR()
        user_input = asr.transcribe(audio_filepath)
    else:
        user_input = text_input

    if not user_input and uploaded_context:
        user_input = "Please read the uploaded document carefully. If it contains a question, answer it. If not, summarize its contents. Ensure you match the language of the document."

    if not user_input:
        yield gr.update(value=""), gr.update(value=None), gr.update(), history, gr.update()
        return

    display_user_input = user_input
    chat_history = []

    for msg in history:
        content = msg.get("content", "")
        if isinstance(content, (dict, list, tuple)): continue
        content_str = str(content)
        if content_str.startswith("{'path':") or "FileData" in content_str: continue

        if msg["role"] == "user":
            chat_history.append(HumanMessage(content=content_str))
        elif msg["role"] == "assistant":
            clean_content = content_str.replace("<div dir='rtl' style='text-align: right;'>\n\n", "").replace("\n\n</div>", "")
            chat_history.append(AIMessage(content=clean_content))

    if uploaded_context:
        if len(uploaded_context) > 15000:
            uploaded_context = uploaded_context[:15000] + "\n... (truncated due to length)"
        chat_history.append(SystemMessage(content=f"The user just uploaded a document. Here is its content:\n{uploaded_context}\n\nPlease use this content to answer the user's query."))

    if file_paths:
        for fp in file_paths:
            actual_path = get_clean_path(fp)
            if os.path.exists(actual_path):
                history.append({"role": "user", "content": gr.FileData(path=actual_path)})

    history.append({"role": "user", "content": display_user_input})
    history.append({"role": "assistant", "content": ""})

    full_answer, docs = "", []
    for chunk, retrieved_docs in pipeline.stream_query(user_input, chat_history=chat_history):
        full_answer += chunk
        docs = retrieved_docs

        display_text = re.sub(r"&lt;think&gt;.*?(&lt;/think&gt;|$)", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
        display_text = re.sub(r"<think[^>]*>.*?(</think[^>]*>|$)", "", display_text, flags=re.DOTALL | re.IGNORECASE).strip()
        history[-1]["content"] = display_text

        # Use gr.update() for audio_in so it doesn't reset/crash mid-stream
        yield gr.update(value=""), gr.update(), gr.update(), history, gr.update()

    is_arabic = bool(re.search("[؀-ۿ]", full_answer))
    voice = "ar-EG-SalmaNeural" if is_arabic else "en-US-AriaNeural"
    tts = HighQualityTTS(voice=voice)

    out_audio = f"response_{uuid.uuid4().hex}.mp3"
    spoken_answer = re.sub(r"&lt;think&gt;.*?&lt;/think&gt;", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
    spoken_answer = re.sub(r"<think[^>]*>.*?</think[^>]*>", "", spoken_answer, flags=re.DOTALL | re.IGNORECASE).strip()

    try:
        if spoken_answer: tts.synthesize(spoken_answer, out_audio)
    except Exception as e:
        logger.error(f"TTS Error: {e}")
        out_audio = None

    answer_text = re.sub(r"&lt;think&gt;.*?(&lt;/think&gt;|$)", "", full_answer, flags=re.DOTALL | re.IGNORECASE)
    answer_text = re.sub(r"<think[^>]*>.*?(</think[^>]*>|$)", "", answer_text, flags=re.DOTALL | re.IGNORECASE).strip()
    history[-1]["content"] = wrap_rtl(answer_text)

    # Clear audio and file inputs once streaming is fully completed
    yield gr.update(value=""), gr.update(value=None), gr.update(value=None), history, out_audio

# Force light mode by overriding the dark theme colors
we_theme = gr.themes.Soft(primary_hue="purple", secondary_hue="indigo").set(
    button_primary_background_fill="#5b2b82",
    button_primary_background_fill_hover="#4a226b",
    button_primary_text_color="white",
    block_title_text_color="#5b2b82"
)

# Load the logo as base64 so it can be embedded directly
try:
    with open("data/we_logo.png", "rb") as f:
        b64_logo = base64.b64encode(f.read()).decode("utf-8")
    logo_html = f"<div style='text-align: center;'><img src='data:image/png;base64,{b64_logo}' width='150' style='display: inline-block;'/></div>"
except Exception:
    logo_html = ""

with gr.Blocks() as demo:
    if logo_html:
        gr.HTML(logo_html)
    gr.Markdown("<h1 style='text-align: center; color: #5b2b82;'>Telecom Egypt Intelligent Assistant</h1>")

    chatbot = gr.Chatbot(label="TE Assistant", height=500)
    with gr.Row():
        gr.HTML("<div style='flex-grow: 1;'></div>")
        with gr.Column(scale=2, min_width=250):
            audio_output = gr.Audio(autoplay=True, interactive=False, elem_id="sleek-audio")
        gr.HTML("<div style='flex-grow: 1;'></div>")

    with gr.Row():
        with gr.Column(scale=8):
            txt = gr.Textbox(show_label=False, placeholder="Type your message here...", container=False)
        with gr.Column(scale=1, min_width=80):
            submit_btn = gr.Button("Send", variant="primary")

    with gr.Row():
        audio_in = gr.Audio(sources=["microphone"], type="filepath", label="Record Voice (Optional)")
        file_in = gr.File(label="Attach Documents (Optional)", file_count="multiple", file_types=[".pdf", ".docx", ".txt", ".png", ".jpg", ".jpeg"], type="filepath")

    submit_btn.click(fn=lambda: gr.update(interactive=False), outputs=[submit_btn]).then(
        fn=process_interaction, inputs=[audio_in, file_in, txt, chatbot], outputs=[txt, audio_in, file_in, chatbot, audio_output]
    ).then(fn=lambda: gr.update(interactive=True), outputs=[submit_btn])

    txt.submit(fn=lambda: gr.update(interactive=False), outputs=[submit_btn]).then(
        fn=process_interaction, inputs=[audio_in, file_in, txt, chatbot], outputs=[txt, audio_in, file_in, chatbot, audio_output]
    ).then(fn=lambda: gr.update(interactive=True), outputs=[submit_btn])

# Gradio 6.0 compatible launch
demo.launch(
    share=True,
    debug=True,
    theme=we_theme,
    css=".gradio-container {max-width: 900px; margin: auto;} #sleek-audio { border-radius: 50px !important; box-shadow: 0px 4px 15px rgba(91, 43, 130, 0.2) !important; border: 2px solid #5b2b82 !important; overflow: hidden; margin-top: 10px; margin-bottom: 15px; } #sleek-audio .label-wrap { display: none !important; }"
)

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3bcef5c1e5995b43f2.gradio.live


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://8f6dd7e79262dc6492.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
